In [ ]:
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# Creating schema for getting structured output everytime

class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed Feedback For Essay")
    score: int = Field(description="Score out of 10", ge=0, le=0)
    

In [ ]:
# Creating model with structured schema

structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
# Defining State of workflow
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


In [ ]:
# evaluate_language Node Logic
def evaluate_language(state: EssayState):
    prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {"language_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
# evaluate_analysis Node Logic
def evaluate_analysis(state: EssayState):
    prompt=f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {"analysis_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
# evaluate_thought Node Logic
def evaluate_thought(state: EssayState):
    prompt=f'Evaluate the clearity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    
    return {"clarity_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
# final_evaluation Node Logic
def final_evaluation(state: EssayState):
    prompt=f'Based on the following feedback create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    
    # We will not use structured_model because we have created different schema for it
    overall_feedback = model.invoke(prompt).content

    # Avg Calculation
    avg_score = sum(state["individual_scores"]) / len(state["individual_scores"])

    return {"overall_feedback": overall_feedback, "avg_score": avg_score}

In [ ]:
# Define Graph
mygraph=StateGraph(EssayState)

# Create Nodes
mygraph.add_node("evaluate_language", evaluate_language)
mygraph.add_node("evaluate_analysis", evaluate_analysis)
mygraph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Create edges
mygraph.add_edge(START, "evaluate_language")
mygraph.add_edge(START, "evaluate_analysis")
mygraph.add_edge(START, "evaluate_thought")

mygraph.add_edge("evaluate_language", "final_evaluation")
mygraph.add_edge("evaluate_analysis", "final_evaluation")
mygraph.add_edge("evaluate_thought", "final_evaluation")

mygraph.add_edge("final_evaluation", END)
workflow=mygraph.compile()


In [ ]:
# Execute the workflow

# The essay text
essay = " "

initial_state={
    "essay": essay,
}

final_state = workflow.invoke(initial_state)